In [1]:
import os

os.chdir("..")

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from catboost import CatBoostRegressor, Pool, cv
import optuna
import mlflow
import mlflow.catboost

/home/cecil/projects/ares/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train = pd.read_csv("artifacts/feature_engineering/features_train.csv")
test = pd.read_csv("artifacts/feature_engineering/features_test.csv")

target = "log_price"

X_train = train.drop(columns=target)
y_train = train[target]

X_test = test.drop(columns=target)
y_test = test[target]


In [4]:
def objective(trial):
    params = {
        "iterations": 4000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10.0, log=True),
        "bootstrap_type": "MVS",
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "random_strength": trial.suggest_float("random_strength", 1.0, 15.0),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 50),
        "grow_policy": "SymmetricTree",
        "random_seed": 42,
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "task_type": "CPU",
        "verbose": False,
        "allow_writing_files": False,
    }

    if params["grow_policy"] == "Lossguide":
        params["max_leaves"] = trial.suggest_int("max_leaves", 16, 64)

    cv_pool = Pool(X_train, y_train)

    with mlflow.start_run(nested=True):
        cv_results = cv(
            params=params,
            dtrain=cv_pool,
            fold_count=5,
            inverted=False,
            shuffle=True,
            partition_random_seed=2026,
            early_stopping_rounds=100,
            verbose=False,
        )

        best_avg_rmse = np.min(cv_results["test-RMSE-mean"])

        mlflow.log_params(params)
        mlflow.log_metric("cv_rmse_mean", best_avg_rmse)
        mlflow.log_metric(
            "cv_rmse_std",
            cv_results["test-RMSE-std"].iloc[np.argmin(cv_results["test-RMSE-mean"])],
        )

    return best_avg_rmse


In [5]:
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("catboost_optuna_ares")


study = optuna.create_study(direction="minimize")
study.enqueue_trial(
    {
        "learning_rate": 0.06099,
        "depth": 6,
        "l2_leaf_reg": 3.0,
        "subsample": 0.8,
        "grow_policy": "SymmetricTree",
        "random_strength": 1.0,
        "min_data_in_leaf": 1,
    }
)
study.optimize(objective, n_trials=10)

print("Best params:", study.best_params)

/home/cecil/projects/ares/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)
[I 2026-02-01 10:48:03,809] A new study created in memory with name: no-name-3095558e-0916-4453-b01d-21b5d155fe94


Training on fold [0/5]

bestTest = 0.4661834591
bestIteration = 542

Training on fold [1/5]

bestTest = 0.4609415782
bestIteration = 685

Training on fold [2/5]

bestTest = 0.4534347958
bestIteration = 652

Training on fold [3/5]

bestTest = 0.446775394
bestIteration = 799

Training on fold [4/5]


[I 2026-02-01 10:49:05,216] Trial 0 finished with value: 0.45664033058179926 and parameters: {'learning_rate': 0.06099, 'depth': 6, 'l2_leaf_reg': 3.0, 'subsample': 0.8, 'random_strength': 1.0, 'min_data_in_leaf': 1}. Best is trial 0 with value: 0.45664033058179926.



bestTest = 0.4546742231
bestIteration = 716

Training on fold [0/5]

bestTest = 0.4869435504
bestIteration = 90

Training on fold [1/5]

bestTest = 0.4659688133
bestIteration = 138

Training on fold [2/5]

bestTest = 0.4535461081
bestIteration = 120

Training on fold [3/5]

bestTest = 0.4566262009
bestIteration = 114

Training on fold [4/5]


[I 2026-02-01 10:49:28,562] Trial 1 finished with value: 0.466910509953607 and parameters: {'learning_rate': 0.23613099079691272, 'depth': 7, 'l2_leaf_reg': 1.402977614369305, 'subsample': 0.9102001104461254, 'random_strength': 1.8214325561056415, 'min_data_in_leaf': 3}. Best is trial 0 with value: 0.45664033058179926.


...


bestTest = 0.4584809057
bestIteration = 2149

Training on fold [0/5]

bestTest = 0.4698286019
bestIteration = 1088

Training on fold [1/5]

bestTest = 0.4615607664
bestIteration = 1711

Training on fold [2/5]

bestTest = 0.4508923218
bestIteration = 1400

Training on fold [3/5]

bestTest = 0.4454379121
bestIteration = 1197

Training on fold [4/5]


[I 2026-02-01 11:02:18,880] Trial 8 finished with value: 0.45708267180402123 and parameters: {'learning_rate': 0.04175087251411997, 'depth': 5, 'l2_leaf_reg': 2.8442602613208554, 'subsample': 0.778128812318441, 'random_strength': 10.062144922087215, 'min_data_in_leaf': 28}. Best is trial 0 with value: 0.45664033058179926.



bestTest = 0.4569471955
bestIteration = 1289

Training on fold [0/5]

bestTest = 0.4772441476
bestIteration = 2844

Training on fold [1/5]

bestTest = 0.4602330757
bestIteration = 3078

Training on fold [2/5]

bestTest = 0.452794996
bestIteration = 2677

Training on fold [3/5]

bestTest = 0.4529968086
bestIteration = 2727

Training on fold [4/5]


[I 2026-02-01 11:07:15,077] Trial 9 finished with value: 0.4601543626967368 and parameters: {'learning_rate': 0.010378536845856804, 'depth': 7, 'l2_leaf_reg': 2.2167849694541726, 'subsample': 0.8835299774689689, 'random_strength': 13.359160839310645, 'min_data_in_leaf': 15}. Best is trial 0 with value: 0.45664033058179926.



bestTest = 0.4574516838
bestIteration = 2697

Best params: {'learning_rate': 0.06099, 'depth': 6, 'l2_leaf_reg': 3.0, 'subsample': 0.8, 'random_strength': 1.0, 'min_data_in_leaf': 1}


In [7]:
print("\nTraining final model with best params...")
best_params = study.best_params

final_params = {
    **best_params,
    "iterations": 3000,
    "random_seed": 2026,
    "verbose": False,
}

final_model = CatBoostRegressor(**final_params)
final_model.fit(
    X_train, y_train, eval_set=[(X_test, y_test)], early_stopping_rounds=100
)

test_preds = final_model.predict(X_test)
final_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
final_r2 = r2_score(y_test, test_preds)

print(f"Best Trial CV RMSE: {study.best_value:.4f}")
print(f"Final Test RMSE: {final_rmse:.4f}")
print(f"Final Test R2 Score: {final_r2:.4f}")



Training final model with best params...


Best Trial CV RMSE: 0.4566
Final Test RMSE: 0.4608
Final Test R2 Score: 0.8513
